<a href="https://colab.research.google.com/github/jen257/cs4207-coding-challenge/blob/main/CS4207_Group4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Setup & Config

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import pyarrow
import sklearn
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed

# 1. Switch data folder automatically (Rockhill vs Google Colab)
if os.path.exists("/home/cs4207/data"):
    DATA_DIR = "/home/cs4207/data"
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/CS4207"

TRAIN_FILES = [
    f"{DATA_DIR}/yellow_tripdata_2026-01.parquet",
    f"{DATA_DIR}/yellow_tripdata_2026-02.parquet",
]
VAL_FILE  = f"{DATA_DIR}/yellow_tripdata_2026-03.parquet"
TEST_FILE = f"{DATA_DIR}/yellow_tripdata_2026-04.parquet"

# 2. Global settings (all in one place)
SEED = 42
WORKERS = [1, 2, 4]
REPEATS = 3

# 3. Print environment info
print("Python:", sys.version.split()[0])
print("CPU cores:", os.cpu_count())
print("pandas:", pd.__version__, "| pyarrow:", pyarrow.__version__, "| sklearn:", sklearn.__version__)

START_TIME = time.perf_counter()

Cell 2 — Data Cleaning & Data Parallelism

In [ ]:
# Columns we need (exclude tip_amount and total_amount to avoid data leakage!)
USE_COLS = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "fare_amount",
    "PULocationID", "DOLocationID", "VendorID", "RatecodeID", "payment_type"
]

def clean_one_file(file_path):
    """Read and clean a single monthly parquet file (1 file = 1 shard)."""
    df = pd.read_parquet(file_path, columns=USE_COLS)

    # TODO (Member 1):
    # 1. Keep only Card (1) and Cash (2) -> payment_type.isin([1, 2])
    # 2. Create binary target: y = 1 for Card, 0 for Cash
    # 3. Filter bad rows: fare_amount > 0, 0 < trip_distance < 100, duration between 1 and 180 mins
    # 4. Drop missing values and return cleaned df
    return df

# Sequential cleaning
def clean_sequential(files):
    dfs = [clean_one_file(f) for f in files]
    return pd.concat(dfs, ignore_index=True)

# Data-parallel cleaning (same function on different shards)
def clean_parallel(files, n_workers=2):
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        dfs = list(pool.map(clean_one_file, files))  # pool.map keeps fixed file order
    return pd.concat(dfs, ignore_index=True)

# TODO (Member 1): Run both on TRAIN_FILES and check if len(df_seq) == len(df_par)

Cell 3 — Task Parallelism A: Feature Engineering

In [ ]:
def get_duration(df):
    """Task 1: Trip duration in minutes."""
    # TODO (Member 2): return DataFrame with duration_min and log_duration
    pass

def get_time_features(df):
    """Task 2: Pickup hour (sin/cos), weekend, rush hour, night."""
    # TODO (Member 2): return DataFrame with time features
    pass

def get_location_features(df, top_zones):
    """Task 3: Airport flags and top-zone one-hot encoding."""
    # TODO (Member 2): return DataFrame with location features
    pass

def build_features_parallel(df, top_zones):
    """Run the 3 feature tasks at the same time."""
    with ProcessPoolExecutor(max_workers=3) as pool:
        f1 = pool.submit(get_duration, df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]])
        f2 = pool.submit(get_time_features, df[["tpep_pickup_datetime"]])
        f3 = pool.submit(get_location_features, df[["PULocationID", "DOLocationID"]], top_zones)

        # Combine in fixed column order
        X = pd.concat([
            df[["trip_distance", "fare_amount", "passenger_count"]].reset_index(drop=True),
            f1.result().reset_index(drop=True),
            f2.result().reset_index(drop=True),
            f3.result().reset_index(drop=True)
        ], axis=1)
    return X

# TODO (Member 2): Find top 15 zones from train_df only, then build X_train, X_val, X_test

Cell 4 — Baseline Model & Task Parallelism B: Hyperparameter Search

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. Sequential Baseline
# TODO (Member 3): Train one default LogisticRegression(random_state=SEED) and print Accuracy & F1

# 2. Task Parallelism: Try different hyperparameters (C values) in parallel
C_VALUES = [0.01, 0.1, 1.0, 10.0]

def train_one_model(c_val):
    """Train one model with a given C value and return its validation F1."""
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=c_val, max_iter=200, random_state=SEED)
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    score = f1_score(y_val, preds)
    return (c_val, score)

def run_tuning_parallel(c_list, n_workers=4):
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        results = list(pool.map(train_one_model, c_list))
    return results

# TODO (Member 3): Pick best C from results, evaluate on X_test, print classification_report

Cell 5 — Benchmarking & Speedup Chart

In [ ]:
times = {}

for w in WORKERS:
    run_times = []
    for r in range(REPEATS):
        t0 = time.perf_counter()
        # TODO (Member 4): Call clean_parallel(TRAIN_FILES, n_workers=w) or run_tuning_parallel(C_VALUES, n_workers=w)
        t1 = time.perf_counter()
        run_times.append(t1 - t0)
    times[w] = np.mean(run_times)

# Calculate Speedup and Efficiency
t1_base = times[1]
speedups = [t1_base / times[w] for w in WORKERS]
efficiencies = [s / w for s, w in zip(speedups, WORKERS)]

# Show table
bench_df = pd.DataFrame({
    "Workers": WORKERS,
    "Mean Time (s)": [times[w] for w in WORKERS],
    "Speedup": speedups,
    "Efficiency": efficiencies
})
print(bench_df)

# Plot Speedup vs Workers
plt.figure(figsize=(6, 4))
plt.plot(WORKERS, speedups, marker="o", label="Actual Speedup")
plt.plot(WORKERS, WORKERS, linestyle="--", label="Ideal Speedup")
plt.xlabel("Number of Workers")
plt.ylabel("Speedup")
plt.title("Parallel Speedup vs Workers")
plt.legend()
plt.grid(True)
plt.show()

Cell 6 — Non-Determinism & Fixes

In [ ]:
# --- Source 1: Unordered worker completion (as_completed vs pool.map) ---
# TODO (Member 5):
# Before fix: Use as_completed() to collect shard results twice -> show row order can differ
# After fix : Use pool.map() (fixed shard order) twice -> assert exact match

# --- Source 2: Unseeded randomness (no random_state vs fixed SEED) ---
# TODO (Member 5):
# Before fix: Run a randomized split/model (e.g., SGDClassifier without random_state) twice -> show different F1/weights
# After fix : Set random_state=SEED twice -> show identical F1/weights

Cell 7 — Total Runtime

In [ ]:
print(f"Total notebook runtime: {time.perf_counter() - START_TIME:.2f} seconds")